# Identity Anomaly Detection System - Documentation

## 1. System Architecture

The system follows a modern **Microservices Architecture** with a decoupled frontend and backend, powered by an ensemble ML engine.

### High-Level Data Flow:
1.  **User/System** generates authentication logs.
2.  **FastAPI Backend** ingests logs and preprocesses them via the Data Pipeline.
3.  **ML Engine** (Ensemble) predicts risk scores using two distinct models.
4.  **React Dashboard** visualizes real-time risks and alerts security analysts.

### Architecture Diagram (Mermaid):

```mermaid
graph TD
    User[User / Auth System] -->|Logs| API[FastAPI Backend]
    API -->|Raw Data| PL[Data Pipeline]
    PL -->|Features| ML[Ensemble ML Engine]
    
    subgraph "ML Engine"
    ML --> IF[Isolation Forest]
    ML --> AE[Autoencoder]
    IF --> Vote[Voting Logic]
    AE --> Vote
    end
    
    Vote -->|Risk Score| DB[(Database / CSV)]
    DB -->|History| API
    API -->|JSON| UI[React Dashboard]
    UI -->|Alerts| Analyst[Security Analyst]
```

## 2. Model Selection Rationale

We utilize an **Ensemble Approach** combining statistical and deep learning models. This ensures robustness against known patterns while maintaining the ability to detect novel, unseen attacks.

### A. Isolation Forest (Statistical Model)
**Why?**
- **Efficient**: Works extremely well on high-dimensional datasets.
- **Unsupervised**: Does not require labeled attack data (which is rare/expensive).
- **Concept**: It isolates anomalies by randomly partitioning data. Anomalies require fewer splits to be isolated than normal points.
- **Best For**: Detecting outliers that are distinctly different from the norm (e.g., Impossible Travel).

### B. Autoencoder (Deep Learning Model)
**Why?**
- **Complex Patterns**: Capable of learning non-linear relationships in authentication behavior.
- **Reconstruction Error**: The model is trained to compress and reconstruct *normal* data. When it sees an attack, it fails to reconstruct it accurately, resulting in a high error (Risk Score).
- **Best For**: Subtle anomalies like Credential Stuffing or slow Lateral Movement.

### C. Ensemble Voting Strategy
**Why?**
- **Reduced False Positives**: By requiring consensus or weighted agreement between models, we reduce false alarms.
- **Holistic Coverage**: IF catches clear statistical outliers; AE catches subtle behavioral shifts.

## 3. Deployment Guide

This guide covers how to set up and run the full stack locally.

### Prerequisites
- **Python 3.8+** (for Backend)
- **Node.js 16+** & **npm** (for Frontend)

### Step 1: Backend Setup (FastAPI)
Navigate to the backend directory and install dependencies:

```bash
cd backend
pip install -r requirements.txt
```

Start the server:
```bash
uvicorn backend.main:app --reload --port 8000
```
*Server will start at `http://localhost:8000`*

### Step 2: Frontend Setup (React)
Navigate to the frontend directory:

```bash
cd frontend
npm install
```

Start the development server:
```bash
npm run dev
```
*UI will be accessible at `http://localhost:5173`*

### Step 3: Verification
- Open the Dashboard at `http://localhost:5173`.
- Check Backend Health: `http://localhost:8000/api/health`.